# 03 · Estimate elasticity and evaluate pricing scenarios

**Questions:** How does the price–demand association change after adding controls, and what do the estimates imply for revenue and approximate gross profit?

**Input:** the analysis Parquet file from notebook 01. **Outputs:** three CSVs in `outputs/python_results/` for notebook 04.

The Cheerios case study uses categorical promotion controls. The ten-product portfolio uses a binary recorded-promotion indicator. Both include store and week effects and use store-clustered uncertainty for their final estimates. These specifications are deliberately distinct and are identified in the exports.

Run top to bottom. Historical numerical findings below describe the original dataset run; revised notebook outputs are cleared until rerun.

## 1. Load data and construct model variables

The sample already has positive quantity and unit price, so both can be logged. In a log–log model, the coefficient on log price is elasticity: an estimate of −1.59 corresponds to roughly 1.59% lower quantity for a small 1% price increase, conditional on the included controls. Missing promotion codes become a separate category rather than causing observations to be dropped.

In [ ]:
from pathlib import Path

# Support kernels started in the repository root or notebooks directory.
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "notebooks").is_dir() or not (PROJECT / "README.md").is_file():
    raise RuntimeError("Start the notebook from the repository root or notebooks directory.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

analysis = pd.read_parquet(
    PROJECT / "data" / "processed" / "cereal_analysis.parquet"
)

cheerios = analysis[
    analysis["UPC"] == 1600066610
].copy()

print(f"Cheerios model sample: {len(cheerios):,} records.")

cheerios["LOG_QTY"] = np.log(cheerios["MOVE"])
cheerios["LOG_PRICE"] = np.log(cheerios["UNIT_PRICE"])

cheerios["SALE_CODE"] = (
    cheerios["SALE"]
    .fillna("NO_CODE")
    .astype("category")
)

## 2. Compare four progressively adjusted models

1. **Price only:** the unadjusted association.
2. **Promotion categories:** account for differences associated with recorded promotion codes.
3. **Store effects:** allow each store its own baseline demand.
4. **Week effects:** also absorb common demand conditions in each week.

One comparison table replaces repeated full regression summaries and intermediate tables. Read changes in elasticity as sensitivity to controls, not proof that the final model is causal. The standard errors in this comparison are conventional OLS errors; the next section supplies clustered uncertainty for the preferred model.

In [ ]:
specifications = {
    "1: Price only": "LOG_QTY ~ LOG_PRICE",
    "2: + Promotion": "LOG_QTY ~ LOG_PRICE + C(SALE_CODE)",
    "3: + Store effects": "LOG_QTY ~ LOG_PRICE + C(SALE_CODE) + C(STORE)",
    "4: + Week effects": "LOG_QTY ~ LOG_PRICE + C(SALE_CODE) + C(STORE) + C(WEEK)",
}
models = {label: smf.ols(formula, data=cheerios).fit()
          for label, formula in specifications.items()}
model_comparison = pd.DataFrame([
    {"Model": label, "Price Elasticity": model.params["LOG_PRICE"],
     "Std. Error (OLS)": model.bse["LOG_PRICE"], "R-squared": model.rsquared,
     "Observations": int(model.nobs)}
    for label, model in models.items()
])
model_comparison

## 3. Estimate uncertainty allowing dependence within stores

Sales observations from one store can share unobserved shocks over time. Cluster standard errors by store for the full specification. This changes uncertainty, not the OLS price coefficient. The confidence interval describes sampling uncertainty under this model; it does not account for all confounding or modeling choices.

In [ ]:
model_4_clustered = smf.ols(
    "LOG_QTY ~ LOG_PRICE + C(SALE_CODE) + C(STORE) + C(WEEK)",
    data=cheerios
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cheerios["STORE"]}
)

print("Price elasticity:",
      model_4_clustered.params["LOG_PRICE"])

print("Clustered standard error:",
      model_4_clustered.bse["LOG_PRICE"])

print("p-value:",
      model_4_clustered.pvalues["LOG_PRICE"])

print("95% confidence interval:")
print(
    model_4_clustered.conf_int().loc["LOG_PRICE"]
)

## 4. Check sensitivity to deep discounts and recorded promotions

First, compare the full sample with samples excluding the bottom 1%, 2.5%, and 5% of observed prices. The original full-sample model is reused rather than fitted again. These exclusions are sensitivity checks only; the primary analysis retains deep discounts.

In [ ]:
price_cutoffs = {
    "Full sample": cheerios["UNIT_PRICE"].min(),
    "Exclude bottom 1%": cheerios["UNIT_PRICE"].quantile(0.01),
    "Exclude bottom 2.5%": cheerios["UNIT_PRICE"].quantile(0.025),
    "Exclude bottom 5%": cheerios["UNIT_PRICE"].quantile(0.05)
}

sensitivity_results = []

for label, cutoff in price_cutoffs.items():

    sample = cheerios[
        cheerios["UNIT_PRICE"] >= cutoff
    ].copy()

    model = model_4_clustered if label == "Full sample" else smf.ols(
        "LOG_QTY ~ LOG_PRICE + C(SALE_CODE) + C(STORE) + C(WEEK)",
        data=sample
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sample["STORE"]}
    )

    sensitivity_results.append({
        "Sample": label,
        "Minimum Price": cutoff,
        "Observations": int(model.nobs),
        "Price Elasticity": model.params["LOG_PRICE"],
        "Clustered SE": model.bse["LOG_PRICE"],
        "p-value": model.pvalues["LOG_PRICE"]
    })

sensitivity_table = pd.DataFrame(sensitivity_results)

sensitivity_table

### Records without a promotion code

Fit the same store and week controls to records labeled `NO_CODE`, omitting a promotion control that would be constant in this subset. This checks dependence on explicitly recorded promotions. It is not a clean “non-promotional” experiment because missing codes do not establish the absence of a promotion.

In [ ]:
no_code = cheerios[
    cheerios["SALE_CODE"] == "NO_CODE"
].copy()

model_no_code = smf.ols(
    "LOG_QTY ~ LOG_PRICE + C(STORE) + C(WEEK)",
    data=no_code
).fit(
    cov_type="cluster",
    cov_kwds={"groups": no_code["STORE"]}
)

print("Observations:", int(model_no_code.nobs))
print("Price elasticity:", model_no_code.params["LOG_PRICE"])
print("Clustered SE:", model_no_code.bse["LOG_PRICE"])
print("95% CI:")
print(model_no_code.conf_int().loc["LOG_PRICE"])

### Interpretation of the original results

The original full-sample Cheerios estimate was about −1.59, compared with −1.56 among records without promotion codes and −1.07 after excluding the lowest 5% of prices. The negative association persisted, but its magnitude depended on the price range. The following scenarios use the full-sample estimate while acknowledging that a constant elasticity may not describe every price range equally well.

## 5. Set a price and approximate cost reference

The dataset's `PROFIT` field is treated as a gross-margin percentage, so implied unit cost is `UNIT_PRICE × (1 − PROFIT / 100)`. It is an accounting-based approximation, not a directly observed economic marginal cost.

Use separate median price and implied cost values from the final 26 weeks. The original reference values were $3.35 and $2.824. Holding cost fixed makes the revenue–profit tradeoff explicit; it does not forecast future costs.

In [ ]:
cheerios["IMPLIED_COST"] = cheerios["UNIT_PRICE"] * (1 - cheerios["PROFIT"] / 100)
last_week = cheerios["WEEK"].max()
recent = cheerios.loc[cheerios["WEEK"] > last_week - 26]
reference_price = recent["UNIT_PRICE"].median()
reference_cost = recent["IMPLIED_COST"].median()
elasticity = model_4_clustered.params["LOG_PRICE"]

print(f"Reference window: weeks {recent['WEEK'].min()}–{recent['WEEK'].max()}")
print(f"Median price: ${reference_price:.3f}; approximate unit cost: ${reference_cost:.3f}")

## 6. Compare five local price scenarios

For price ratio `r = scenario price / reference price` and elasticity `e`, quantity is indexed as `r ** e` and revenue as `r ** (1 + e)`. Approximate gross profit is indexed as `(scenario price − cost) × quantity index / (reference price − cost)`.

An index of 1 is the reference outcome. Percentage changes below are relative to that reference, not absolute sales forecasts. Revenue and profit are calculated together once, replacing the separate revenue-only scenario table. Export values retain full precision; rounding is only for display.

In [ ]:
profit_scenarios = pd.DataFrame({
    "Price Change": [-0.10, -0.05, 0.00, 0.05, 0.10]
})

profit_scenarios["Scenario Price"] = (
    reference_price *
    (1 + profit_scenarios["Price Change"])
)

profit_scenarios["Quantity Index"] = (
    (1 + profit_scenarios["Price Change"]) ** elasticity
)

profit_scenarios["Revenue Index"] = (
    (1 + profit_scenarios["Price Change"]) *
    profit_scenarios["Quantity Index"]
)

profit_scenarios["Gross Profit per Unit"] = (
    profit_scenarios["Scenario Price"] -
    reference_cost
)

baseline_gp_per_unit = (
    reference_price -
    reference_cost
)

profit_scenarios["Gross Profit Index"] = (
    profit_scenarios["Gross Profit per Unit"] *
    profit_scenarios["Quantity Index"]
    / baseline_gp_per_unit
)

profit_scenarios["Revenue Change"] = (
    profit_scenarios["Revenue Index"] - 1
)

profit_scenarios["Gross Profit Change"] = (
    profit_scenarios["Gross Profit Index"] - 1
)

profit_display = profit_scenarios.copy()

profit_display["Price Change"] = (
    profit_display["Price Change"] * 100
).round(1)

profit_display["Scenario Price"] = (
    profit_display["Scenario Price"]
).round(2)

profit_display["Quantity Change"] = (
    (profit_display["Quantity Index"] - 1) * 100
).round(2)

profit_display["Revenue Change"] = (
    profit_display["Revenue Change"] * 100
).round(2)

profit_display["Gross Profit Change"] = (
    profit_display["Gross Profit Change"] * 100
).round(2)

profit_display[
    [
        "Price Change",
        "Scenario Price",
        "Quantity Change",
        "Revenue Change",
        "Gross Profit Change"
    ]
]

## 7. Examine the wider price grid

Evaluate the original one-cent grid, starting at $2.75 and ending near the sample's maximum observed price. The grid shows how a constant-elasticity, fixed-cost scenario behaves beyond the five local changes. A maximum at the grid boundary is not an identified optimal price. Negative gross-profit indices indicate a simulated price below the assumed cost.

In [ ]:
price_grid = np.arange(
    2.75,
    cheerios["UNIT_PRICE"].max() + 0.001,
    0.01
)

pricing_grid = pd.DataFrame({
    "Scenario Price": price_grid
})

pricing_grid["Price Ratio"] = (
    pricing_grid["Scenario Price"] /
    reference_price
)

pricing_grid["Quantity Index"] = (
    pricing_grid["Price Ratio"] ** elasticity
)

pricing_grid["Revenue Index"] = (
    pricing_grid["Price Ratio"] *
    pricing_grid["Quantity Index"]
)

pricing_grid["Gross Profit per Unit"] = (
    pricing_grid["Scenario Price"] -
    reference_cost
)

pricing_grid["Gross Profit Index"] = (
    pricing_grid["Gross Profit per Unit"] *
    pricing_grid["Quantity Index"]
    / baseline_gp_per_unit
)

best_profit_row = pricing_grid.loc[
    pricing_grid["Gross Profit Index"].idxmax()
]

best_revenue_row = pricing_grid.loc[
    pricing_grid["Revenue Index"].idxmax()
]

print("Model-implied gross-profit maximum:")
print(best_profit_row)

print("\nHighest modeled revenue in grid:")
print(best_revenue_row)

plt.figure(figsize=(10, 6))

plt.plot(
    pricing_grid["Scenario Price"],
    pricing_grid["Revenue Index"],
    label="Revenue"
)

plt.plot(
    pricing_grid["Scenario Price"],
    pricing_grid["Gross Profit Index"],
    label="Approx. Gross Profit"
)

plt.axvline(
    reference_price,
    linestyle="--",
    label=f"Reference Price (${reference_price:.2f})"
)

plt.axhline(
    1,
    linestyle=":"
)

plt.xlabel("Scenario Price ($)")
plt.ylabel("Index (Reference Price = 1.00)")
plt.title("Modeled Revenue and Gross Profit by Price")
plt.legend()

plt.show()

### Business interpretation

In the original results, a modeled 5% price increase reduced revenue by about 2.9% while increasing approximate gross profit by 22%. Gross profit continued increasing to the tested grid's upper boundary. This illustrates a tradeoff, not a proven profit-maximizing price.

A controlled test of a moderate increase could evaluate actual demand response before a broader pricing decision. Observational elasticity, constant demand response, and fixed approximate cost remain limitations. The retained CSVs and dashboard contain the original results; rerunning this notebook recalculates them.

## 8. Estimate the ten-product portfolio

Select the ten products with highest historical revenue before fitting their models. This avoids choosing products based on their resulting elasticity. Each product gets its own price coefficient, store and week effects, and store-clustered uncertainty.

Unlike the Cheerios scenario model, the portfolio uses a binary `SALE.notna()` indicator for whether a code was recorded. This preserves the original specification and avoids assigning economic meanings to sparse categories. Pandas' source CSV parsing handles empty fields as missing; a literal whitespace code is not normalized here. SQL reporting separately trims whitespace in its reporting flag.

Failed fits are reported and block export, so an incomplete portfolio cannot silently replace the ten-product output.

In [ ]:
top_products = (
    analysis.groupby(["UPC", "DESCRIP", "SIZE"])
    .agg(
        total_revenue=("REVENUE", "sum"),
        total_units=("MOVE", "sum"),
        observations=("MOVE", "size")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

portfolio_results = []
failed_products = []

for _, product in top_products.iterrows():

    upc = product["UPC"]

    product_data = analysis[
        analysis["UPC"] == upc
    ].copy()

    product_data["LOG_QTY"] = np.log(
        product_data["MOVE"]
    )

    product_data["LOG_PRICE"] = np.log(
        product_data["UNIT_PRICE"]
    )

    product_data["PROMO_RECORDED"] = (
        product_data["SALE"].notna().astype(int)
    )

    product_data = product_data[
        np.isfinite(product_data["LOG_QTY"]) &
        np.isfinite(product_data["LOG_PRICE"])
    ].copy()

    try:

        model = smf.ols(
            """
            LOG_QTY ~ LOG_PRICE
            + PROMO_RECORDED
            + C(STORE)
            + C(WEEK)
            """,
            data=product_data
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": product_data["STORE"]
            }
        )

        ci = model.conf_int().loc["LOG_PRICE"]

        portfolio_results.append({
            "UPC": upc,
            "Product": product["DESCRIP"],
            "Size": product["SIZE"],
            "Observations": int(model.nobs),
            "Stores": product_data["STORE"].nunique(),
            "Weeks": product_data["WEEK"].nunique(),
            "Elasticity": model.params["LOG_PRICE"],
            "Clustered SE": model.bse["LOG_PRICE"],
            "CI Lower": ci.iloc[0],
            "CI Upper": ci.iloc[1],
            "R Squared": model.rsquared
        })

    except Exception as e:

        failed_products.append({
            "UPC": upc,
            "Product": product["DESCRIP"],
            "Size": product["SIZE"],
            "Error": str(e)
        })

portfolio_elasticities = pd.DataFrame(
    portfolio_results
)

failed_products = pd.DataFrame(
    failed_products
)

if not failed_products.empty:
    print(failed_products.to_string(index=False))
    raise RuntimeError("Some product models failed; resolve these errors before exporting a partial portfolio.")

## 9. Read estimates alongside confidence intervals

The original results included eight point estimates below −1. The exported demand and revenue-effect labels are point-estimate summaries retained for dashboard compatibility; they do not encode statistical certainty. When a confidence interval crosses −1, the elastic/inelastic distinction is uncertain. Exactly −1 is the unit-elastic boundary, although the existing binary export labels place it in the other group.

For negative elasticities, an estimate below −1 implies quantity changes proportionally more than price and a local price increase reduces modeled revenue. Values between −1 and zero imply the opposite revenue direction. Any non-negative estimate would require separate scrutiny rather than a routine “inelastic demand” interpretation.

In [ ]:
portfolio_elasticities["Demand Type"] = np.where(
    portfolio_elasticities["Elasticity"] < -1,
    "Elastic",
    "Inelastic"
)

portfolio_elasticities["Revenue Effect of Price Increase"] = np.where(
    portfolio_elasticities["Elasticity"] < -1,
    "Revenue likely decreases",
    "Revenue likely increases"
)

portfolio_elasticities[["Product", "Size", "Elasticity", "CI Lower", "CI Upper", "Demand Type"]]

## 10. Export results for SQL reporting

Write the portfolio estimates, five scenarios, and detailed grid using the existing column names and filenames. UPC is exported as text for joining. `model_id` distinguishes the binary-promotion portfolio from the categorical-promotion Cheerios scenarios; reference price, cost, and elasticity travel with both scenario tables so the assumptions remain visible.

Notebook 04 imports these files into DuckDB and prepares the Power BI tables.

In [ ]:
EXPORTS = PROJECT / "outputs" / "python_results"
EXPORTS.mkdir(parents=True, exist_ok=True)

def standardize_columns(df):
    result = df.copy()
    result.columns = (
        result.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return result

# Results for your ten modeled products
elasticities_export = standardize_columns(portfolio_elasticities)
elasticities_export["upc"] = elasticities_export["upc"].astype(str)
elasticities_export["model_id"] = "portfolio_binary_promo"

# Five Cheerios pricing scenarios
scenarios_export = standardize_columns(profit_scenarios)
scenarios_export["upc"] = "1600066610"
scenarios_export["model_id"] = "cheerios_categorical_promo"

# Detailed Cheerios price grid
grid_export = standardize_columns(pricing_grid)
grid_export["upc"] = "1600066610"
grid_export["model_id"] = "cheerios_categorical_promo"

# Save the assumptions with both scenario tables
for table in [scenarios_export, grid_export]:
    table["reference_price"] = float(reference_price)
    table["reference_cost"] = float(reference_cost)
    table["elasticity"] = float(
        model_4_clustered.params["LOG_PRICE"]
    )

elasticities_export.to_csv(
    EXPORTS / "product_elasticities.csv", index=False
)
scenarios_export.to_csv(
    EXPORTS / "pricing_scenarios.csv", index=False
)
grid_export.to_csv(
    EXPORTS / "pricing_grid.csv", index=False
)

print("Exports saved to:", EXPORTS)
print("Products:", len(elasticities_export))
print("Scenarios:", len(scenarios_export))
print("Grid points:", len(grid_export))